In [1]:
import pandas as pd
from mlxtend import frequent_patterns
from mlxtend.preprocessing import TransactionEncoder

import service
from association import apriori

In [2]:
df = pd.read_json("data/data_1.jsonl", lines=True)  # TODO change to variant
df.head()

,Transaction
0,"[0, 1, 2, 5]"
1,"[1, 2, 5]"
2,"[1, 3, 4, 6, 7]"
3,"[1, 2, 4, 5, 6, 9]"
4,"[1, 2, 3, 4, 5]"


In [3]:
transactions = df['Transaction'].to_list()
num_transactions = len(transactions)

num_transactions

10

#### Apriori algorithm

In [4]:
min_support = 4 / num_transactions  # TODO change to variant
min_confidence = 0.0

In [5]:
frequent_itemsets = apriori.apriori_algorithm(transactions, min_support=min_support)

for itemset, count in frequent_itemsets.items():
    print(f"{set(itemset)}: {count}")

{'0'}: 4
{'1'}: 6
{'2'}: 4
{'5'}: 4
{'3'}: 5
{'4'}: 5
{'6'}: 4
{'7'}: 4
{'1', '2'}: 4
{'5', '1'}: 4
{'4', '1'}: 4
{'5', '2'}: 4
{'5', '1', '2'}: 4


In [6]:
rules = apriori.association_rules(frequent_itemsets, transactions, min_confidence=min_confidence)

for antecedent, consequent, confidence in rules:
    print(f"{set(antecedent)} => {set(consequent)} ({confidence:.2f})")

{'1'} => {'2'} (0.67)
{'2'} => {'1'} (1.00)
{'5'} => {'1'} (1.00)
{'1'} => {'5'} (0.67)
{'4'} => {'1'} (0.80)
{'1'} => {'4'} (0.67)
{'5'} => {'2'} (1.00)
{'2'} => {'5'} (1.00)
{'5'} => {'1', '2'} (1.00)
{'1'} => {'5', '2'} (0.67)
{'2'} => {'5', '1'} (1.00)
{'5', '1'} => {'2'} (1.00)
{'5', '2'} => {'1'} (1.00)
{'1', '2'} => {'5'} (1.00)


Compare with similar `mlxtend` algorithm

In [7]:
encoder = TransactionEncoder()
te_ary = encoder.fit_transform(transactions)
mlxtend_df = pd.DataFrame(te_ary, columns=encoder.columns_)

In [8]:
mlxtend_frequent_itemsets = frequent_patterns.apriori(
    mlxtend_df, min_support=min_support, use_colnames=True
)

for _, (support, itemset) in mlxtend_frequent_itemsets.iterrows():
    count = int(support * num_transactions)
    print(f"{set(itemset)}: {count}")

{'0'}: 4
{'1'}: 6
{'2'}: 4
{'3'}: 5
{'4'}: 5
{'5'}: 4
{'6'}: 4
{'7'}: 4
{'1', '2'}: 4
{'4', '1'}: 4
{'5', '1'}: 4
{'5', '2'}: 4
{'5', '1', '2'}: 4


In [9]:
mlxtend_rules = frequent_patterns.association_rules(
    mlxtend_frequent_itemsets, metric="confidence", min_threshold=min_confidence
)[["antecedents", "consequents", "confidence"]]

mlxtend_rules

,antecedents,consequents,confidence
0,(1),(2),0.666667
1,(2),(1),1.000000
2,(4),(1),0.800000
3,(1),(4),0.666667
4,(5),(1),1.000000
5,(1),(5),0.666667
6,(5),(2),1.000000
7,(2),(5),1.000000
8,"(5, 1)",(2),1.000000
9,"(5, 2)",(1),1.000000


In [10]:
assert service.check_frequent_itemsets_equal(
    frequent_itemsets, mlxtend_frequent_itemsets, num_transactions
)
assert service.check_rules_equal(rules, mlxtend_rules)